# Personalized LeetCode Recommender
**CS-31 Final Project, Preliminary Workflow** by Siri Vadlamudi & Kabir Goklani

---

## Workflow Overview

### What did we make?
We made a notebook that reads your LeetCode practice history (a CSV you can export or just keep by hand) and recommends the next 5 problems you should practice, along with a short reason for each one. Fixed lists like Blind 75 give everyone the same order no matter what they already know, so we wanted something that actually adapts to you. Our recommender looks at three things. First, it tracks how well you seem to know each topic, using a Bayesian estimate that also shows how uncertain it is instead of just giving you a single grade. Second, it pays attention to how long ago you practiced each topic, and older attempts count for less because people forget stuff. Third, it considers how many days you have until your interview, so topics you haven't touched in a while get more weight as the deadline gets closer.

### How did we make it?
To figure out how well you know each topic, we gave every topic a Beta–Bernoulli belief (Section 3). Each topic starts at a `Beta(1, 1)` prior, and every solve adds to α while every failed attempt adds to β. We also wanted the model to account for forgetting, so in Section 3 we decay the evidence exponentially, meaning an attempt from `d` days ago counts `0.5 ** (d / HALF_LIFE_DAYS)`, which is an idea we got from Duolingo's half-life regression. To pick which topics to recommend, Section 4 uses Thompson sampling, which is a multi-armed bandit method. It samples a plausible mastery level from each topic's Beta distribution and favors topics that are either weak *or* uncertain, and then it adds a "staleness" bonus on top that grows as your interview gets closer. Finally, to pick the actual problems, Section 4 pulls unsolved problems from a curated problem bank (in the style of Blind 75 and NeetCode) and also mixes in retries of problems you failed before.

Everything is written in plain Python with `pandas`, `numpy`, and `matplotlib`. There are no accounts and no scraping, and nothing gets uploaded anywhere, so your CSV stays on your own machine.

### Example
Say Maya has her Google interview in 2 weeks. She's been keeping a CSV of what she's practiced, and she's pretty solid on Arrays but failed two Graphs problems last week. To use the recommender, she replaces `data/practice_history.csv` with her own file, which just needs the same four columns (`problem`, `topic`, `date`, and `solved`). Then she runs all the cells and types `14` when it asks how many days she has until her interview. Section 3 shows that Graphs has a low estimated mastery with a wide uncertainty band, and Section 5 gives her five problems that are mostly Graphs and DP, each with a reason like *"needs more practice (est. 32%)"* and a link to the problem on LeetCode. After she solves a few of them, she adds new rows to her CSV and re-runs the notebook to get an updated list.

### How to modify it for your needs
All of the settings you can change are in the Configuration cell in Section 1, including the input file paths, the forgetting half-life, the number of recommendations, the prior, and the default interview deadline. Section 6 ("Make it yours") re-runs the recommender with different settings so you can actually see what each one does. If you want to add topics or problems, you can just add rows to `data/problem_bank.csv`, and any new `topic` value gets picked up automatically.

### Credits, prior art & AI-use disclosure
Our candidate problems come from public curated lists (Blind 75, NeetCode 150, and Grind 75), and every recommendation links back to the original problem on LeetCode. A few published techniques inspired our approach. Bayesian Knowledge Tracing (Corbett & Anderson, pyBKT) inspired the per-topic belief, Duolingo's half-life regression (Settles & Meeder, 2016) inspired the forgetting decay, and Clement et al. (2015, arXiv:1310.3174) inspired the bandit-based way we pick topics. We didn't copy any code from these sources, and the Beta–Bernoulli and Thompson sampling code here is our own standard implementation. We used Claude Code, an AI coding assistant, to help draft this notebook's code, alongside our own code and changes based on our project proposal and design decisions. We reviewed, ran, and tested all of the code ourselves.

## 1. Setup & configuration
Run this cell first, this is the main place to customize the workflow.

In [1]:
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

# ============================== CONFIGURATION ==============================
HISTORY_PATH = Path("data/practice_history.csv")  # your practice log
BANK_PATH = Path("data/problem_bank.csv")          # candidate problems to recommend

TODAY = date.today()               # or pin it for reproducible output, e.g. date(2026, 11, 23)
DEFAULT_DAYS_TO_INTERVIEW = 21     # used if you press Enter at the prompt (or in automated runs)
HALF_LIFE_DAYS = 14                # an attempt this many days old counts half as much
N_RECS = 5                         # how many problems to recommend
PRIOR_ALPHA, PRIOR_BETA = 1.0, 1.0 # Beta(1,1) = "no idea yet": a cautious, uniform prior
SEED = 31                          # fixes the randomness so re-runs give the same list
# ===========================================================================

print(f"Today: {TODAY} | half-life: {HALF_LIFE_DAYS} days | recommendations: {N_RECS}")

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load and check the data

**Practice history** (`HISTORY_PATH`) needs one row per attempt:

| column | meaning | example |
|---|---|---|
| `problem` | LeetCode problem title | `Two Sum` |
| `topic` | topic tag (any name you like) | `Arrays` |
| `date` | date of the attempt, `YYYY-MM-DD` | `2026-11-01` |
| `solved` | `1` if you solved it unaided, `0` if not | `1` |

**Problem bank** (`BANK_PATH`) lists candidate problems: `problem, topic, url`.

The loaders below check the columns and give a clear message if something is wrong, instead of failing later with a confusing error.

In [17]:
HISTORY_COLUMNS = ["problem", "topic", "date", "solved"]
BANK_COLUMNS = ["problem", "topic", "url"]


def _require_columns(df: pd.DataFrame, required: list, file_label: str) -> None:
    """Raise a readable error if a CSV is missing required columns."""
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{file_label} is missing column(s) {missing}. Found: {list(df.columns)}")


def load_history(path: Path) -> pd.DataFrame:
    """Load the practice log: strip whitespace, parse dates, and coerce `solved` to 0/1."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    _require_columns(df, HISTORY_COLUMNS, str(path))
    df["problem"] = df["problem"].astype(str).str.strip()
    df["topic"] = df["topic"].astype(str).str.strip()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date
    df["solved"] = pd.to_numeric(df["solved"], errors="coerce")

    bad_rows = df["date"].isna() | ~df["solved"].isin([0, 1])
    if bad_rows.any():
        print(f"Skipping {bad_rows.sum()} row(s) with an unreadable date or a `solved` value other than 0/1:")
        display(df[bad_rows])
    df = df[~bad_rows].copy()
    df["solved"] = df["solved"].astype(int)
    return df.sort_values("date").reset_index(drop=True)


def load_problem_bank(path: Path) -> pd.DataFrame:
    """Load candidate problems and drop duplicate titles."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    _require_columns(df, BANK_COLUMNS, str(path))
    for col in BANK_COLUMNS:
        df[col] = df[col].astype(str).str.strip()
    return df.drop_duplicates(subset="problem").reset_index(drop=True)


history = load_history(HISTORY_PATH)
problem_bank = load_problem_bank(BANK_PATH)

print(f"{len(history)} attempts across {history['topic'].nunique()} topics; "
      f"{len(problem_bank)} problems in the bank.")
display(history)

10 attempts across 4 topics; 20 problems in the bank.


,problem,topic,date,solved
0,Two Sum,Arrays,2026-11-01,1
1,Climbing Stairs,Dynamic Programming,2026-11-03,0
2,Number of Islands,Graphs,2026-11-05,0
3,Maximum Depth of Binary Tree,Trees,2026-11-07,1
4,Valid Anagram,Arrays,2026-11-09,1
5,House Robber,Dynamic Programming,2026-11-12,0
6,Clone Graph,Graphs,2026-11-15,0
7,Invert Binary Tree,Trees,2026-11-16,1
8,Contains Duplicate,Arrays,2026-11-17,1
9,Coin Change,Dynamic Programming,2026-11-19,1


### When is your interview?
The prompt accepts a date (`2026-12-14`) or a number of days (`14`). If you press entire, type something unreadable, or run the notebook non-interactively (for example with *Run All* in an autograder), it uses `DEFAULT_DAYS_TO_INTERVIEW`. It never blocks or crashes.

In [3]:
def parse_interview_answer(answer: str, today: date, default_days: int) -> int:
    """Turn the user's answer (a date, a day count, or blank) into days until the interview."""
    answer = (answer or "").strip()
    if not answer:
        return default_days
    if answer.isdigit():
        return max(int(answer), 0)
    try:
        interview_day = datetime.strptime(answer, "%Y-%m-%d").date()
        return max((interview_day - today).days, 0)
    except ValueError:
        print(f"Couldn't read {answer!r}; using the default of {default_days} days.")
        return default_days


def ask_days_to_interview(today: date, default_days: int) -> int:
    """Ask the user, falling back to the default when no one can answer (e.g. headless runs)."""
    try:
        answer = input(f"Interview date (YYYY-MM-DD) or days until interview [Enter = {default_days}]: ")
    except Exception:  # no stdin available (nbconvert, autograder, etc.)
        answer = ""
    return parse_interview_answer(answer, today, default_days)


DAYS_TO_INTERVIEW = ask_days_to_interview(TODAY, DEFAULT_DAYS_TO_INTERVIEW)
print(f"Planning for an interview in {DAYS_TO_INTERVIEW} day(s).")

# Quick self-check of the parser (also documents the accepted formats):
assert parse_interview_answer("", date(2026, 11, 23), 21) == 21
assert parse_interview_answer("10", date(2026, 11, 23), 21) == 10
assert parse_interview_answer("2026-12-01", date(2026, 11, 23), 21) == 8

Planning for an interview in 14 day(s).


## 3. Estimating topic mastery

For each topic, we keep a Beta(α, β) distribution over "the probability you'd solve a new problem in this topic today." Every topic starts at the prior `Beta(PRIOR_ALPHA, PRIOR_BETA)`, and with the default `Beta(1, 1)` every mastery level is equally likely, so a topic you haven't practiced at all shows up as uncertain instead of looking like you've already mastered it. Then for each attempt you made `d` days ago, we compute a forgetting weight `w = 0.5 ** (d / HALF_LIFE_DAYS)`. If you solved the problem we add that weight to α (`α += w`), and if you didn't solve it we add it to β (`β += w`). The mean `α / (α + β)` is our estimate of how well you know the topic, and the 80% credible interval shows how sure we are about that estimate, so a wide band basically means "we don't know yet."

Since older attempts get discounted, a topic you aced a month ago will slowly drift back toward "uncertain" if you stop practicing it, which is pretty much what happens in real life too.

In [11]:
def forgetting_weight(days_ago, half_life: float):
    """How much an attempt `days_ago` old still counts (1.0 = today, 0.5 = one half-life ago)."""
    return 0.5 ** (np.asarray(days_ago, dtype=float) / half_life)


In [12]:
@dataclass
class TopicBelief:
    """Our current belief about one topic's mastery."""
    topic: str
    alpha: float
    beta: float
    attempts: int
    days_since_practice: float  # np.inf if never practiced

    @property
    def mean(self) -> float:
        return self.alpha / (self.alpha + self.beta)

    @property
    def evidence(self) -> float:
        """Effective number of (decayed) attempts behind this estimate."""
        return self.alpha + self.beta - PRIOR_ALPHA - PRIOR_BETA

    def credible_interval(self, rng: np.random.Generator, level: float = 0.80, n_samples: int = 20_000):
        samples = rng.beta(self.alpha, self.beta, n_samples)
        tail = (1 - level) / 2
        return float(np.quantile(samples, tail)), float(np.quantile(samples, 1 - tail))


def describe_mastery(belief: TopicBelief) -> str:
    """Neutral, non-judgmental label. On purpose, it never says 'ready' or 'mastered'."""
    if belief.evidence < 1.0:
        return "not enough history yet"
    if belief.mean < 0.5:
        return "needs more practice"
    if belief.mean < 0.7:
        return "making progress"
    return "going well so far"


def estimate_topic_beliefs(history: pd.DataFrame, topics: list, today: date,
                           half_life: float) -> dict:
    """Build a decayed Beta belief for every topic (topics with no history keep the prior)."""
    beliefs = {}
    for topic in topics:
        attempts = history[history["topic"] == topic]
        days_ago = np.array([(today - d).days for d in attempts["date"]], dtype=float)
        days_ago = np.clip(days_ago, 0, None)  # ignore accidental future dates
        weights = forgetting_weight(days_ago, half_life)
        solved = attempts["solved"].to_numpy()
        beliefs[topic] = TopicBelief(
            topic=topic,
            alpha=PRIOR_ALPHA + float(np.sum(weights * solved)),
            beta=PRIOR_BETA + float(np.sum(weights * (1 - solved))),
            attempts=len(attempts),
            days_since_practice=float(days_ago.min()) if len(attempts) else np.inf,
        )
    return beliefs


def beliefs_table(beliefs: dict, seed: int) -> pd.DataFrame:
    """Summarize beliefs as a sorted table (weakest topic first)."""
    rng = np.random.default_rng(seed)
    rows = []
    for b in beliefs.values():
        low, high = b.credible_interval(rng)
        rows.append({
            "topic": b.topic, "attempts": b.attempts, "decayed evidence": round(b.evidence, 2),
            "est. mastery": round(b.mean, 2), "80% low": round(low, 2), "80% high": round(high, 2),
            "days since practice": b.days_since_practice, "status": describe_mastery(b),
        })
    return pd.DataFrame(rows).sort_values("est. mastery").reset_index(drop=True)


ALL_TOPICS = sorted(set(history["topic"]) | set(problem_bank["topic"]))
beliefs = estimate_topic_beliefs(history, ALL_TOPICS, TODAY, HALF_LIFE_DAYS)
mastery = beliefs_table(beliefs, SEED)
display(mastery)

,topic,attempts,decayed evidence,est. mastery,80% low,80% high,days since practice,status
0,Graphs,2,1.08,0.32,0.05,0.67,8.0,needs more practice
1,Dynamic Programming,3,1.77,0.48,0.17,0.80,4.0,needs more practice
2,Trees,2,1.16,0.68,0.34,0.95,7.0,making progress
3,Arrays,3,1.58,0.72,0.41,0.96,6.0,going well so far


The table above is sorted with your weakest topic first. A low "est. mastery" means the model thinks that topic needs more work, and a big gap between "80% low" and "80% high" means it doesn't have much evidence yet, so treat the number as a rough guess. These are just estimates from a handful of solved/unsolved flags, so they aren't grading you and they don't say whether you're "ready."

## 4. Choosing what to practice (Thompson-sampling bandit)

We treat each topic like an "arm" of a multi-armed bandit, which is basically a row of slot machines where you have to decide which lever to pull. Here, pulling an arm means practicing that topic, and the payoff is how much you're likely to learn, which is highest for topics where your mastery is low.

To fill each of the `N_RECS` slots, we first sample a plausible mastery level `θ ~ Beta(α, β)` for every topic. Topics we're unsure about have wide distributions, so sometimes they sample low and get picked even if their average looks fine, and that's how Thompson sampling balances exploration (trying topics we don't know much about yet) with exploitation (focusing on topics we already know are weak). Next we score each topic with `score = (1 − θ) + urgency × staleness`. The `1 − θ` part is how much room you have to improve. The `staleness` part goes from 0 to 1 and measures how long it's been since you last practiced the topic, compared to `2 × HALF_LIFE_DAYS`. The `urgency` part is `1 / (1 + days_to_interview / 7)`, which comes out to about 0.25 when your interview is three weeks away and about 0.7 when it's three days away, since reviewing topics you haven't touched in a while matters more the closer you get. Finally, we pick the highest-scoring topic that still has problems left and take its next problem.

The candidate problems for each topic come from two places. First are problems you attempted but never solved, which we bring back as retries, and after those come problems from the bank that you haven't solved yet, in the bank's curated order.

In [7]:
def leetcode_url(title: str) -> str:
    """Best-guess LeetCode URL from a title, for history problems that aren't in the bank."""
    slug = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")
    return f"https://leetcode.com/problems/{slug}/"


def build_candidates(history: pd.DataFrame, problem_bank: pd.DataFrame) -> dict:
    """Map topic -> ordered list of candidate problems (retries first, then unseen bank problems)."""
    solved_titles = set(history.loc[history["solved"] == 1, "problem"])
    bank_urls = dict(zip(problem_bank["problem"], problem_bank["url"]))

    candidates = {}
    failed = history[(history["solved"] == 0) & ~history["problem"].isin(solved_titles)]
    for _, row in failed.drop_duplicates("problem").iterrows():
        candidates.setdefault(row["topic"], []).append({
            "problem": row["problem"], "topic": row["topic"], "is_retry": True,
            "url": bank_urls.get(row["problem"], leetcode_url(row["problem"])),
        })
    for _, row in problem_bank[~problem_bank["problem"].isin(solved_titles)].iterrows():
        already_listed = {c["problem"] for c in candidates.get(row["topic"], [])}
        if row["problem"] not in already_listed:
            candidates.setdefault(row["topic"], []).append({
                "problem": row["problem"], "topic": row["topic"], "is_retry": False, "url": row["url"],
            })
    return candidates


def deadline_urgency(days_to_interview: int) -> float:
    """0 to 1: rises as the interview approaches."""
    return 1.0 / (1.0 + days_to_interview / 7.0)


def staleness(belief: TopicBelief, half_life: float) -> float:
    """0 = practiced today, 1 = not practiced for 2 half-lives (or never)."""
    return float(min(belief.days_since_practice / (2 * half_life), 1.0))


def explain(belief: TopicBelief, candidate: dict, stale: float, urgency: float) -> str:
    """Plain-language reason for a recommendation (shown to the student)."""
    reasons = []
    if candidate["is_retry"]:
        reasons.append("retry of a problem you didn't solve before")
    if belief.evidence < 1.0:
        reasons.append("little recent history, so the estimate is uncertain")
    elif belief.mean < 0.5:
        reasons.append(f"needs more practice (est. {belief.mean:.0%})")
    if stale >= 0.5 and np.isfinite(belief.days_since_practice):
        reasons.append(f"not practiced in {int(belief.days_since_practice)} days")
    if urgency >= 0.5 and stale >= 0.3:
        reasons.append("interview is close, so worth a refresh")
    if not reasons:
        reasons.append(f"keeps the topic fresh (est. {belief.mean:.0%})")
    return "; ".join(reasons)


def recommend(beliefs: dict, candidates: dict, days_to_interview: int, n_recs: int,
              half_life: float, seed: int) -> pd.DataFrame:
    """Pick `n_recs` problems with Thompson sampling plus a deadline-aware staleness bonus."""
    rng = np.random.default_rng(seed)
    urgency = deadline_urgency(days_to_interview)
    remaining = {topic: list(problems) for topic, problems in candidates.items()}
    picks = []

    for slot in range(1, n_recs + 1):
        scores = {}
        for topic, belief in beliefs.items():
            if not remaining.get(topic):
                continue  # nothing left to recommend in this topic
            theta = rng.beta(belief.alpha, belief.beta)
            scores[topic] = (1 - theta) + urgency * staleness(belief, half_life)
        if not scores:
            print(f"Only {len(picks)} candidate problem(s) left; add more to the problem bank.")
            break
        topic = max(scores, key=scores.get)
        candidate = remaining[topic].pop(0)
        belief = beliefs[topic]
        stale = staleness(belief, half_life)
        picks.append({
            "#": slot, "problem": candidate["problem"], "topic": topic,
            "why": explain(belief, candidate, stale, urgency),
            "score": round(scores[topic], 2), "url": candidate["url"],
        })
    return pd.DataFrame(picks)


candidates = build_candidates(history, problem_bank)
print({topic: len(problems) for topic, problems in candidates.items()})

{'Dynamic Programming': 7, 'Graphs': 7, 'Arrays': 4, 'Trees': 5}


## 5. Your next problems

In [14]:
def show_recommendations(recs: pd.DataFrame) -> None:
    """Render recommendations with clickable LeetCode links."""
    table = recs.copy()
    table["problem"] = [f'<a href="{u}" target="_blank">{p}</a>' for p, u in zip(table["problem"], table["url"])]
    display(HTML(table.drop(columns="url").to_html(index=False, escape=False)))


recommendations = recommend(beliefs, candidates, DAYS_TO_INTERVIEW, N_RECS, HALF_LIFE_DAYS, SEED)
print(f"Interview in {DAYS_TO_INTERVIEW} days (urgency = {deadline_urgency(DAYS_TO_INTERVIEW):.2f})")
show_recommendations(recommendations)

Interview in 14 days (urgency = 0.33)


#,problem,topic,why,score
1,Number of Islands,Graphs,retry of a problem you didn't solve before; needs more practice (est. 32%),1.09
2,Climbing Stairs,Dynamic Programming,retry of a problem you didn't solve before; needs more practice (est. 48%),0.53
3,Clone Graph,Graphs,retry of a problem you didn't solve before; needs more practice (est. 32%),1.07
4,Course Schedule,Graphs,needs more practice (est. 32%),0.70
5,Rotting Oranges,Graphs,needs more practice (est. 32%),0.80


These are just suggestions, not a quota, so feel free to skip anything you don't want to do. Once you've practiced, add a row to your CSV for each attempt (with `solved` set to 1 or 0) and re-run the notebook, and the estimates and recommendations will update based on what you did.

## 6. Making it yours

Since every step above is its own function, you can re-run the whole thing with different settings without touching the code. Here are three changes you'd probably want to try.

### (a) Interview in 3 days
If your interview is only 3 days away, urgency goes way up, so topics you haven't practiced in a while start moving up the list.

In [9]:
recs_soon = recommend(beliefs, candidates, days_to_interview=3, n_recs=N_RECS,
                      half_life=HALF_LIFE_DAYS, seed=SEED)
show_recommendations(recs_soon)

#,problem,topic,why,score
1,Number of Islands,Graphs,retry of a problem you didn't solve before; needs more practice (est. 32%),1.20
2,Climbing Stairs,Dynamic Programming,retry of a problem you didn't solve before; needs more practice (est. 48%),0.59
3,Clone Graph,Graphs,retry of a problem you didn't solve before; needs more practice (est. 32%),1.17
4,Course Schedule,Graphs,needs more practice (est. 32%),0.81
5,Rotting Oranges,Graphs,needs more practice (est. 32%),0.90


### (b) Slower forgetting (30-day half-life)
If you think you forget stuff more slowly, you can bump the half-life up to 30 days. Older attempts then count for more, so the estimates move further away from the prior.

In [15]:
slow_forget_beliefs = estimate_topic_beliefs(history, ALL_TOPICS, TODAY, half_life=30)
display(beliefs_table(slow_forget_beliefs, SEED)[["topic", "est. mastery", "80% low", "80% high", "status"]])
show_recommendations(recommend(slow_forget_beliefs, candidates, DAYS_TO_INTERVIEW, N_RECS, 30, SEED))

,topic,est. mastery,80% low,80% high,status
0,Graphs,0.29,0.04,0.61,needs more practice
1,Dynamic Programming,0.44,0.16,0.74,needs more practice
2,Trees,0.72,0.40,0.96,going well so far
3,Arrays,0.76,0.49,0.97,going well so far


#,problem,topic,why,score
1,Number of Islands,Graphs,retry of a problem you didn't solve before; needs more practice (est. 29%),1.04
2,Climbing Stairs,Dynamic Programming,retry of a problem you didn't solve before; needs more practice (est. 44%),0.56
3,Clone Graph,Graphs,retry of a problem you didn't solve before; needs more practice (est. 29%),1.02
4,House Robber,Dynamic Programming,retry of a problem you didn't solve before; needs more practice (est. 44%),0.72
5,Course Schedule,Graphs,needs more practice (est. 29%),0.79


### (c) Logging a new attempt
Say you just solved *Course Schedule* (Graphs) today. Normally you'd just add a row to your CSV, but here we add it in code so you can see how the estimate and the recommendations change.

In [11]:
new_attempt = pd.DataFrame([{"problem": "Course Schedule", "topic": "Graphs", "date": TODAY, "solved": 1}])
updated_history = pd.concat([history, new_attempt], ignore_index=True)

updated_beliefs = estimate_topic_beliefs(updated_history, ALL_TOPICS, TODAY, HALF_LIFE_DAYS)
before, after = beliefs["Graphs"], updated_beliefs["Graphs"]
print(f"Graphs mastery estimate: {before.mean:.2f} -> {after.mean:.2f} "
      f"(evidence {before.evidence:.2f} -> {after.evidence:.2f})")
show_recommendations(recommend(updated_beliefs, build_candidates(updated_history, problem_bank),
                               DAYS_TO_INTERVIEW, N_RECS, HALF_LIFE_DAYS, SEED))

Graphs mastery estimate: 0.32 -> 0.49 (evidence 1.08 -> 2.08)


#,problem,topic,why,score
1,Climbing Stairs,Dynamic Programming,retry of a problem you didn't solve before; needs more practice (est. 48%),0.81
2,House Robber,Dynamic Programming,retry of a problem you didn't solve before; needs more practice (est. 48%),0.56
3,House Robber II,Dynamic Programming,needs more practice (est. 48%),0.73
4,Unique Paths,Dynamic Programming,needs more practice (est. 48%),0.69
5,Decode Ways,Dynamic Programming,needs more practice (est. 48%),0.72
